# Entrenamiento y evaluación de modelos para predecir la variable **default** 

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path().resolve().parent.parent / "data/data-09-2025"
data_file = "cleaned_data_default.parquet"

df = pd.read_parquet(DATA_DIR / data_file)
df.head()

,plazo,vinculacion,v_cuota,v_prestamo,s_capital,s_intereses,aportes,garantias,valorgarantia,ctasahorros,...,actividadeconomica,estado_cliente,departamento,sexo,curtotalingresos,curtotalegresos,intestrato,actualizacion,default,puntaje_data
n_credito,,,,,,,,,,,,,,,,,,,,,
003-002-0125852-7,1827,8103,356849.0,15000000.0,12923538.0,123855,7741255,1,7741255,33042953.0,...,asalariados,1,antioquia,0,4597000.0,1500000.0,5.0,1,0,795.0
004-002-0068475-5,1826,1434,2650409.0,100460000.0,31911361.0,263265,4601706,1,4601706,3791115.0,...,asalariados,1,antioquia,0,4597000.0,650000.0,5.0,1,0,836.0
003-002-0122592-9,1826,573,791482.0,30000000.0,23844684.0,261477,530431,1,530431,94435.0,...,asalariados,1,antioquia,0,4400000.0,2000000.0,4.0,0,1,709.0
006-002-0023879-0,2922,1902,2860501.0,176000000.0,113842595.0,1008570,3023534,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,733.0
006-002-0026159-4,2557,1902,987637.0,50300000.0,38521256.0,317167,1023082,2,320385440,54841.0,...,educacion_basica_secundaria,1,antioquia,0,22020000.0,1500000.0,4.0,1,0,695.0


In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12860 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12860 non-null  int64   
 1   vinculacion         12860 non-null  int64   
 2   v_cuota             12860 non-null  float64 
 3   v_prestamo          12860 non-null  float64 
 4   s_capital           12860 non-null  float64 
 5   s_intereses         12860 non-null  int64   
 6   aportes             12860 non-null  int64   
 7   garantias           12860 non-null  int64   
 8   valorgarantia       12860 non-null  int64   
 9   ctasahorros         12860 non-null  float64 
 10  edad                12860 non-null  float64 
 11  tipoasociado        12860 non-null  int64   
 12  actividadeconomica  12851 non-null  category
 13  estado_cliente      12860 non-null  int64   
 14  departamento        12859 non-null  category
 15  sexo         

In [18]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12850 entries, 003-002-0125852-7 to 003-002-0119478-4
Data columns (total 22 columns):
 #   Column              Non-Null Count  Dtype   
---  ------              --------------  -----   
 0   plazo               12850 non-null  int64   
 1   vinculacion         12850 non-null  int64   
 2   v_cuota             12850 non-null  float64 
 3   v_prestamo          12850 non-null  float64 
 4   s_capital           12850 non-null  float64 
 5   s_intereses         12850 non-null  int64   
 6   aportes             12850 non-null  int64   
 7   garantias           12850 non-null  int64   
 8   valorgarantia       12850 non-null  int64   
 9   ctasahorros         12850 non-null  float64 
 10  edad                12850 non-null  float64 
 11  tipoasociado        12850 non-null  int64   
 12  actividadeconomica  12850 non-null  category
 13  estado_cliente      12850 non-null  int64   
 14  departamento        12850 non-null  category
 15  sexo         

In [19]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["default", "s_capital"]) # s_capital está muy correlacionada con v_prestamo
y = df["default"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

print(f"Training set size: {X_train.shape}")
print(f"Testing set size: {X_test.shape}")

Training set size: (10280, 20)
Testing set size: (2570, 20)


In [20]:
from sklearn.metrics import classification_report, confusion_matrix


def print_resultados(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(model.feature_importances_, index=X_train.columns)
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [21]:
import lightgbm as lgb
from lightgbm import LGBMClassifier
from sklearn.metrics import f1_score

model = LGBMClassifier(
    objective="binary",
    class_weight="balanced",
    verbose=0,
    random_state=1,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
)

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    categorical_feature=cat_vars,
    eval_set=[(val_x, val_y)],
    eval_metric="logloss",
    callbacks=[lgb.early_stopping(stopping_rounds=20)],
)

train_score = f1_score(y_train, model.predict(X_train))
test_score = f1_score(y_test, model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[100]	valid_0's binary_logloss: 0.172171
Train score: 0.89
Test score: 0.76


In [22]:
print_resultados(model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.93      0.95      2125
           1       0.71      0.83      0.76       445

    accuracy                           0.91      2570
   macro avg       0.84      0.88      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1972  153]
 [  75  370]] 

10 características más importantes:
s_intereses         432
vinculacion         356
valorgarantia       272
puntaje_data        254
curtotalingresos    241
ctasahorros         235
v_cuota             199
plazo               181
v_prestamo          177
aportes             173
dtype: int32


In [23]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import TargetEncoder
from xgboost import XGBClassifier

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

train_x, val_x, train_y, val_y = train_test_split(
    X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
)

model = XGBClassifier(
    grow_policy="lossguide",
    tree_method="hist",
    early_stopping_rounds=20,
    eval_metric="auc",
    random_state=1,
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
)

model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.96
Test score: 0.79


In [24]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      2125
           1       0.78      0.79      0.79       445

    accuracy                           0.93      2570
   macro avg       0.87      0.87      0.87      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2027   98]
 [  94  351]] 

10 características más importantes:
actualizacion       0.492536
estado_cliente      0.118741
tipoasociado        0.055485
garantias           0.036793
edad                0.033694
puntaje_data        0.027630
v_prestamo          0.026591
curtotalingresos    0.025893
s_intereses         0.023650
valorgarantia       0.021748
dtype: float32


In [25]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=1, max_depth=5)
model.fit(X_train_processed, y_train)
train_score = f1_score(y_train, model.predict(X_train_processed))
test_score = f1_score(y_test, model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.69
Test score: 0.64


In [26]:
print_resultados(model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.92      0.95      0.93      2125
           1       0.70      0.59      0.64       445

    accuracy                           0.88      2570
   macro avg       0.81      0.77      0.79      2570
weighted avg       0.88      0.88      0.88      2570
 

matriz de confusión:
[[2010  115]
 [ 182  263]] 

10 características más importantes:
actualizacion       0.280928
tipoasociado        0.253261
edad                0.160095
garantias           0.104524
v_prestamo          0.068483
aportes             0.062069
curtotalingresos    0.046228
s_intereses         0.017328
estado_cliente      0.004058
curtotalegresos     0.001823
dtype: float64


In [27]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    auto_class_weights='Balanced',
    verbose=0,
    )

train_x, val_x, train_y, val_y = train_test_split(
    X_train, 
    y_train, 
    test_size=0.2, 
    stratify=y_train, 
    random_state=1
    )

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

model.fit(
    train_x,
    train_y,
    cat_features=cat_vars,
    eval_set=(val_x, val_y),
    early_stopping_rounds=20,
    )

train_score = f1_score(y_train, cat_model.predict(X_train))
test_score = f1_score(y_test, cat_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

Train score: 0.81
Test score: 0.76


In [28]:
print_resultados(model, X_test_na, y_test_na)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.97      0.91      0.94      2125
           1       0.68      0.87      0.76       445

    accuracy                           0.91      2570
   macro avg       0.82      0.89      0.85      2570
weighted avg       0.92      0.91      0.91      2570
 

matriz de confusión:
[[1941  184]
 [  59  386]] 

10 características más importantes:
actualizacion       12.836181
vinculacion         12.610403
s_intereses         12.052804
tipoasociado        10.212950
ctasahorros          9.324600
puntaje_data         7.275255
valorgarantia        6.127920
curtotalingresos     5.091863
v_cuota              4.634942
aportes              4.533479
dtype: float64


# Sintonización de modelos

### LightGBM

In [37]:
import lightgbm as lgb
import optuna


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        categorical_feature=categorical_features,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

lgb_model.fit(
    X_train, 
    y_train,
    categorical_feature=categorical_features,
    )

lgb_params = lgb_model.get_params()

train_score = f1_score(y_train, lgb_model.predict(X_train))
test_score = f1_score(y_test, lgb_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 15:18:08,398] A new study created in memory with name: no-name-d51c83d5-e248-497e-a846-aa03f007cad2
[I 2026-03-04 15:18:08,521] Trial 0 finished with value: 0.7844155844155845 and parameters: {'lambda_l1': 1.416427688258913, 'lambda_l2': 0.010293096430436202, 'num_leaves': 80, 'feature_fraction': 0.9980038832228971, 'bagging_fraction': 0.46649598595846187, 'bagging_freq': 4, 'min_child_samples': 83, 'learning_rate': 0.041357801826571505, 'colsample_bytree': 0.6482159643066518, 'scale_pos_weight': 3.411448658364523}. Best is trial 0 with value: 0.7844155844155845.
[I 2026-03-04 15:18:08,642] Trial 1 finished with value: 0.7903871829105474 and parameters: {'lambda_l1': 5.775173757145952e-07, 'lambda_l2': 2.2685128878039188e-06, 'num_leaves': 299, 'feature_fraction': 0.6500366295625861, 'bagging_fraction': 0.4896658064130016, 'bagging_freq': 2, 'min_child_samples': 75, 'learning_rate': 0.21395190788697244, 'colsample_bytree': 0.9670190354732457, 'scale_pos_weight': 8.7056259

Best trial: 0.837 with params: {'lambda_l1': 4.0315252621389366e-05, 'lambda_l2': 8.812496315211004e-06, 'num_leaves': 109, 'feature_fraction': 0.8731220772738956, 'bagging_fraction': 0.8067650338666644, 'bagging_freq': 3, 'min_child_samples': 51, 'learning_rate': 0.1455073542070791, 'colsample_bytree': 0.6551843478239394, 'scale_pos_weight': 3.1625546269550813}
Train score: 1.00
Test score: 0.79


In [38]:
print_resultados(lgb_model, X_test, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.96      0.96      2125
           1       0.81      0.78      0.79       445

    accuracy                           0.93      2570
   macro avg       0.88      0.87      0.88      2570
weighted avg       0.93      0.93      0.93      2570
 

matriz de confusión:
[[2044   81]
 [  98  347]] 

10 características más importantes:
s_intereses         1217
vinculacion         1135
curtotalingresos    1086
ctasahorros         1001
v_cuota              920
puntaje_data         870
valorgarantia        855
aportes              745
edad                 703
v_prestamo           685
dtype: int32


### XGBoost

In [31]:
def objective(trial):
    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    # early_stopping_rounds=20,
    # eval_metric="auc",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

xgb_model.fit(X_train_processed, y_train)
xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 14:27:45,792] A new study created in memory with name: no-name-6243f8e4-333b-4616-94db-69eb09de467f
[I 2026-03-04 14:27:46,188] Trial 0 finished with value: 0.8035982008995503 and parameters: {'max_depth': 3, 'learning_rate': 0.10333843408329595, 'n_estimators': 261, 'subsample': 0.7282471014046252, 'colsample_bytree': 0.7871575566707606, 'reg_alpha': 1.4489472056104593, 'reg_lambda': 30.838299064294876, 'max_delta_step': 1, 'gamma': 0.7690699372631418, 'min_child_weight': 8, 'scale_pos_weight': 9.990939879668156}. Best is trial 0 with value: 0.8035982008995503.
[I 2026-03-04 14:27:46,334] Trial 1 finished with value: 0.7987711213517665 and parameters: {'max_depth': 10, 'learning_rate': 0.3186727244673596, 'n_estimators': 318, 'subsample': 0.9587304028459291, 'colsample_bytree': 0.8508372461274407, 'reg_alpha': 0.07986412789797775, 'reg_lambda': 13.389559658564643, 'max_delta_step': 9, 'gamma': 1.0160779776195712, 'min_child_weight': 6, 'scale_pos_weight': 7.8007744075275

Best trial: 0.820 with params: {'max_depth': 12, 'learning_rate': 0.09083138044519092, 'n_estimators': 345, 'subsample': 0.7069221390983305, 'colsample_bytree': 0.6871128155242189, 'reg_alpha': 0.1679141483584423, 'reg_lambda': 0.19965922367977837, 'max_delta_step': 3, 'gamma': 1.3761972818750912, 'min_child_weight': 7, 'scale_pos_weight': 7.406036284955626}
Train score: 0.98
Test score: 0.79


In [32]:
print_resultados(xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95      2125
           1       0.77      0.80      0.79       445

    accuracy                           0.92      2570
   macro avg       0.86      0.88      0.87      2570
weighted avg       0.93      0.92      0.93      2570
 

matriz de confusión:
[[2019  106]
 [  88  357]] 

10 características más importantes:
actualizacion       0.528467
estado_cliente      0.113864
tipoasociado        0.038564
garantias           0.031145
curtotalingresos    0.027203
v_prestamo          0.025647
edad                0.025071
puntaje_data        0.023970
valorgarantia       0.023625
s_intereses         0.021619
dtype: float32


### Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier


def objective(trial):
    te = TargetEncoder()

    cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

    preprocessor = ColumnTransformer(
        transformers=[
            ("target_encoder", te, cat_vars),
        ],
        remainder="passthrough",
    )

    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_test_processed = preprocessor.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

te = TargetEncoder()

cat_vars = X_train.select_dtypes(include=["category"]).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("target_encoder", te, cat_vars),
    ],
    remainder="passthrough",
)

X_train_processed = preprocessor.fit_transform(X_train, y_train)
X_test_processed = preprocessor.transform(X_test)

rf_model.fit(X_train_processed, y_train)
rf_params = rf_model.get_params()

train_score = f1_score(y_train, rf_model.predict(X_train_processed))
test_score = f1_score(y_test, rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-03-04 14:29:27,327] A new study created in memory with name: no-name-9e040643-55b5-423a-83dd-9ee912283fce
[I 2026-03-04 14:29:28,657] Trial 0 finished with value: 0.7483443708609272 and parameters: {'n_estimators': 69, 'max_depth': 14, 'min_samples_split': 9, 'min_samples_leaf': 7, 'max_features': 'log2'}. Best is trial 0 with value: 0.7483443708609272.
[I 2026-03-04 14:29:29,899] Trial 1 finished with value: 0.7487520798668885 and parameters: {'n_estimators': 450, 'max_depth': 7, 'min_samples_split': 10, 'min_samples_leaf': 16, 'max_features': 'sqrt'}. Best is trial 1 with value: 0.7487520798668885.
[I 2026-03-04 14:29:31,046] Trial 2 finished with value: 0.7504132231404959 and parameters: {'n_estimators': 375, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 2 with value: 0.7504132231404959.
[I 2026-03-04 14:29:32,143] Trial 3 finished with value: 0.759075907590759 and parameters: {'n_estimators': 438, 'max_depth': 14, 'min_

Best trial: 0.767 with params: {'n_estimators': 440, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 6, 'max_features': 'log2'}
Train score: 0.80
Test score: 0.75


In [34]:
print_resultados(rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      2125
           1       0.67      0.83      0.75       445

    accuracy                           0.90      2570
   macro avg       0.82      0.87      0.84      2570
weighted avg       0.91      0.90      0.91      2570
 

matriz de confusión:
[[1946  179]
 [  74  371]] 

10 características más importantes:
actualizacion       0.224592
tipoasociado        0.147562
garantias           0.134360
curtotalingresos    0.083871
puntaje_data        0.067580
edad                0.066187
v_prestamo          0.065910
estado_cliente      0.057107
valorgarantia       0.044726
s_intereses         0.034770
dtype: float64


### CatBoost

In [44]:
def objective(trial):
    param = {
        'iterations': trial.suggest_int('iterations', 50, 1000),
        'depth': trial.suggest_int('depth', 3, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 1, 50),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bylevel': trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        'border_count': trial.suggest_int('border_count', 32, 255),
        'random_strength': trial.suggest_float('random_strength', 1e-3, 10, log=True),
        }

    categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = CatBoostClassifier(
        auto_class_weights='Balanced',
        verbose=0,
        )


    model.fit(
        train_x,
        train_y,
        cat_features=categorical_features,
        eval_set=(val_x, val_y),
        early_stopping_rounds=20,
        )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

[I 2026-03-04 16:28:59,816] A new study created in memory with name: no-name-bc22cc29-19a1-4ab4-a136-e3fbfb06e769
[I 2026-03-04 16:29:12,228] Trial 0 finished with value: 0.7875 and parameters: {'iterations': 122, 'depth': 3, 'learning_rate': 0.22455588796952752, 'l2_leaf_reg': 6.28725021581604, 'min_child_samples': 23, 'subsample': 0.6973623761630489, 'colsample_bylevel': 0.7357551252714265, 'border_count': 146, 'random_strength': 0.3480783752629587}. Best is trial 0 with value: 0.7875.
[I 2026-03-04 16:29:34,712] Trial 1 finished with value: 0.7875 and parameters: {'iterations': 193, 'depth': 5, 'learning_rate': 0.010164760608119676, 'l2_leaf_reg': 0.018326415199130194, 'min_child_samples': 7, 'subsample': 0.7565120057144695, 'colsample_bylevel': 0.9040876278759394, 'border_count': 203, 'random_strength': 0.001002188507806512}. Best is trial 0 with value: 0.7875.
[I 2026-03-04 16:30:02,646] Trial 2 finished with value: 0.7875 and parameters: {'iterations': 580, 'depth': 6, 'learning_

Best trial: 0.787 with params: {'iterations': 122, 'depth': 3, 'learning_rate': 0.22455588796952752, 'l2_leaf_reg': 6.28725021581604, 'min_child_samples': 23, 'subsample': 0.6973623761630489, 'colsample_bylevel': 0.7357551252714265, 'border_count': 146, 'random_strength': 0.3480783752629587}


In [47]:
categorical_features = X_train.select_dtypes(include=["category"]).columns.tolist()
print(categorical_features)

['actividadeconomica', 'departamento']


In [48]:
cat_model = CatBoostClassifier(
    **study.best_trial.params,
    auto_class_weights='Balanced',
    verbose=0,
    )

cat_model.fit(
    X_train, 
    y_train,
    cat_features=categorical_features,
    )

cat_model.fit(X_train, y_train)
cat_params = cat_model.get_params()

train_score = f1_score(y_train, cat_model.predict(X_train))
test_score = f1_score(y_test, cat_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

CatBoostError: features data: pandas.DataFrame column 'actividadeconomica' has dtype 'category' but is not in  cat_features list

In [42]:
categorical_features

['actividadeconomica', 'departamento']

# Modelos con Skrub

### LightGBM

In [22]:
import warnings

from skrub import TableVectorizer

warnings.filterwarnings("ignore")


def objective(trial):
    param = {
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-8, 10.0, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-8, 10.0, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 50, 500),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.4, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 7),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = LGBMClassifier(
        **param,
        objective="binary",
        force_col_wise=True,
        random_state=1,
        verbose=-1,
    )

    model.fit(
        train_x,
        train_y,
        eval_set=[(val_x, val_y)],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=False)],
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_lgb_model = LGBMClassifier(
    **study.best_trial.params,
    objective="binary",
    force_col_wise=True,
    random_state=1,
)

sk_lgb_model.fit(X_train_processed, y_train)
sk_lgb_params = sk_lgb_model.get_params()

train_score = f1_score(y_train, sk_lgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_lgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-02-27 13:48:52,247] A new study created in memory with name: no-name-de018806-1abd-40ec-8668-2f8d4478b601
[I 2026-02-27 13:48:52,785] Trial 0 finished with value: 0.7650429799426934 and parameters: {'lambda_l1': 1.858443096815006, 'lambda_l2': 2.825712734833322e-07, 'num_leaves': 477, 'feature_fraction': 0.4006095189841784, 'bagging_fraction': 0.4945934471952099, 'bagging_freq': 6, 'min_child_samples': 39, 'learning_rate': 0.12869849658602894, 'colsample_bytree': 0.7094029049184427, 'scale_pos_weight': 2.1527692787186266}. Best is trial 0 with value: 0.7650429799426934.
[I 2026-02-27 13:48:53,431] Trial 1 finished with value: 0.7677852348993288 and parameters: {'lambda_l1': 0.004192926850648609, 'lambda_l2': 4.498872007937633, 'num_leaves': 304, 'feature_fraction': 0.6186520842736475, 'bagging_fraction': 0.9312724518631427, 'bagging_freq': 1, 'min_child_samples': 68, 'learning_rate': 0.11198932926710332, 'colsample_bytree': 0.9002060571219471, 'scale_pos_weight': 4.830510043756

Best trial: 0.782 with params: {'lambda_l1': 0.00048822227687979575, 'lambda_l2': 0.27533681806134896, 'num_leaves': 409, 'feature_fraction': 0.8045439036265845, 'bagging_fraction': 0.6365149680610661, 'bagging_freq': 5, 'min_child_samples': 13, 'learning_rate': 0.04519984633119446, 'colsample_bytree': 0.8471487702438674, 'scale_pos_weight': 9.956998549962037}
Train score: 0.96
Test score: 0.77


In [23]:
def print_resultados_skrub(model, X_test, y_test):
    print("reporte de clasificación:")
    print(classification_report(y_test, model.predict(X_test)), "\n")
    print("matriz de confusión:")
    print(confusion_matrix(y_test, model.predict(X_test)), "\n")
    feature_importances = pd.Series(
        model.feature_importances_, index=model.feature_names_in_
    )
    feature_importances.sort_values(ascending=False, inplace=True)
    print("10 características más importantes:")
    print(feature_importances.head(10))
    return

In [24]:
print_resultados_skrub(sk_lgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.95      0.95      2126
           1       0.77      0.78      0.77       446

    accuracy                           0.92      2572
   macro avg       0.86      0.86      0.86      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2022  104]
 [  99  347]] 

10 características más importantes:
Column_6     2559
Column_3     2042
Column_15    1967
Column_10    1656
Column_19    1651
Column_9     1644
Column_4     1522
Column_7     1421
Column_5     1296
Column_11    1254
dtype: int32


### Skrub con modelo por defecto

In [25]:
from skrub import tabular_pipeline


def objective(trial):
    param = {
        "histgradientboostingclassifier__learning_rate": trial.suggest_float(
            "histgradientboostingclassifier__learning_rate", 0.01, 0.2
        ),  # noqa: E501
        "histgradientboostingclassifier__max_iter": trial.suggest_int(
            "histgradientboostingclassifier__max_iter", 100, 500
        ),  # noqa: E501
        "histgradientboostingclassifier__max_leaf_nodes": trial.suggest_int(
            "histgradientboostingclassifier__max_leaf_nodes", 20, 100
        ),  # noqa: E501
        "histgradientboostingclassifier__min_samples_leaf": trial.suggest_int(
            "histgradientboostingclassifier__min_samples_leaf", 1, 10
        ),  # noqa: E501
        "histgradientboostingclassifier__l2_regularization": trial.suggest_float(
            "histgradientboostingclassifier__l2_regularization", 1e-3, 10.0, log=True
        ),  # noqa: E501
        "histgradientboostingclassifier__early_stopping": trial.suggest_categorical(
            "histgradientboostingclassifier__early_stopping", [True, False]
        ),  # noqa: E501
    }

    train_x, val_x, train_y, val_y = train_test_split(
        X_train, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = tabular_pipeline(estimator="classifier")

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_params = study.best_trial.params
sk_params = {
    k.replace("histgradientboostingclassifier__", ""): v for k, v in sk_params.items()
}  # noqa: E501
sk_model = tabular_pipeline(estimator="classifier")
sk_model["histgradientboostingclassifier"].set_params(**sk_params)
sk_model.fit(X_train, y_train)
sk_params = sk_model.get_params()

train_score = f1_score(y_train, sk_model.predict(X_train))
test_score = f1_score(y_test, sk_model.predict(X_test))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-02-27 13:49:27,041] A new study created in memory with name: no-name-06cebd41-3d2a-44f2-bd0a-9e5529d2c4d0
[I 2026-02-27 13:49:27,966] Trial 0 finished with value: 0.754601226993865 and parameters: {'histgradientboostingclassifier__learning_rate': 0.16171068735190514, 'histgradientboostingclassifier__max_iter': 454, 'histgradientboostingclassifier__max_leaf_nodes': 27, 'histgradientboostingclassifier__min_samples_leaf': 2, 'histgradientboostingclassifier__l2_regularization': 9.796757471517228, 'histgradientboostingclassifier__early_stopping': False}. Best is trial 0 with value: 0.754601226993865.
[I 2026-02-27 13:49:28,956] Trial 1 finished with value: 0.7537993920972644 and parameters: {'histgradientboostingclassifier__learning_rate': 0.046631411300876106, 'histgradientboostingclassifier__max_iter': 158, 'histgradientboostingclassifier__max_leaf_nodes': 27, 'histgradientboostingclassifier__min_samples_leaf': 4, 'histgradientboostingclassifier__l2_regularization': 0.366093961595

Best trial: 0.764 with params: {'histgradientboostingclassifier__learning_rate': 0.08162364483443112, 'histgradientboostingclassifier__max_iter': 328, 'histgradientboostingclassifier__max_leaf_nodes': 65, 'histgradientboostingclassifier__min_samples_leaf': 5, 'histgradientboostingclassifier__l2_regularization': 1.9332259803220864, 'histgradientboostingclassifier__early_stopping': True}
Train score: 0.96
Test score: 0.75


In [26]:
print("reporte de clasificación:")
print(classification_report(y_test, sk_model.predict(X_test)), "\n")
print("matriz de confusión:")
print(confusion_matrix(y_test, sk_model.predict(X_test)), "\n")

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.94      0.97      0.95      2126
           1       0.83      0.69      0.75       446

    accuracy                           0.92      2572
   macro avg       0.88      0.83      0.85      2572
weighted avg       0.92      0.92      0.92      2572
 

matriz de confusión:
[[2061   65]
 [ 137  309]] 



### XGBoost

In [27]:
def objective(trial):
    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    param = {
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-2, 5e-1, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1e2, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 1e2, log=True),
        "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
        "gamma": trial.suggest_float("gamma", 0, 2),
        "min_child_weight": trial.suggest_int("min_child_weight", 5, 10),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1, 10),
    }

    model = XGBClassifier(
        grow_policy="lossguide",
        tree_method="hist",
        early_stopping_rounds=20,
        eval_metric="auc",
        random_state=1,
    )

    model.fit(train_x, train_y, eval_set=[(val_x, val_y)], verbose=False)

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_xgb_model = XGBClassifier(
    **study.best_trial.params,
    grow_policy="lossguide",
    tree_method="hist",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_xgb_model.fit(X_train_processed, y_train)
sk_xgb_params = xgb_model.get_params()

train_score = f1_score(y_train, sk_xgb_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_xgb_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-02-27 13:50:22,323] A new study created in memory with name: no-name-1f594fa7-8374-47c3-a532-a97530cfa76e
[I 2026-02-27 13:50:23,204] Trial 0 finished with value: 0.7496251874062968 and parameters: {'max_depth': 10, 'learning_rate': 0.2794941935892296, 'n_estimators': 464, 'subsample': 0.5338810500653588, 'colsample_bytree': 0.6633597420453726, 'reg_alpha': 0.004076962468344133, 'reg_lambda': 57.37752269882255, 'max_delta_step': 8, 'gamma': 0.5041604428819548, 'min_child_weight': 7, 'scale_pos_weight': 6.780346024729854}. Best is trial 0 with value: 0.7496251874062968.
[I 2026-02-27 13:50:23,955] Trial 1 finished with value: 0.7481146304675717 and parameters: {'max_depth': 15, 'learning_rate': 0.071440606129705, 'n_estimators': 459, 'subsample': 0.7699973232668278, 'colsample_bytree': 0.6409219163191793, 'reg_alpha': 0.035750444989945666, 'reg_lambda': 55.11938447950022, 'max_delta_step': 5, 'gamma': 0.7980273860856719, 'min_child_weight': 7, 'scale_pos_weight': 8.9850859748913

Best trial: 0.770 with params: {'max_depth': 6, 'learning_rate': 0.07661034086287112, 'n_estimators': 262, 'subsample': 0.5983320547870085, 'colsample_bytree': 0.7118497974161865, 'reg_alpha': 75.76133939930779, 'reg_lambda': 0.005388737043345122, 'max_delta_step': 4, 'gamma': 1.985878292874109, 'min_child_weight': 9, 'scale_pos_weight': 8.654132974538792}
Train score: 0.72
Test score: 0.69


In [28]:
print_resultados_skrub(sk_xgb_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.98      0.85      0.91      2126
           1       0.56      0.91      0.69       446

    accuracy                           0.86      2572
   macro avg       0.77      0.88      0.80      2572
weighted avg       0.91      0.86      0.87      2572
 

matriz de confusión:
[[1810  316]
 [  42  404]] 

10 características más importantes:
actualizacion       0.289205
tipoasociado        0.067057
ctasahorros         0.065190
s_intereses         0.041519
curtotalingresos    0.032261
valorgarantia       0.032184
puntaje_data        0.031301
vinculacion         0.025074
v_cuota             0.023635
aportes             0.022371
dtype: float32


### Random Forest

In [29]:
from sklearn.ensemble import RandomForestClassifier


def objective(trial):
    vectorizer = TableVectorizer()

    X_train_processed = vectorizer.fit_transform(X_train, y_train)
    X_test_processed = vectorizer.transform(X_test)

    train_x, val_x, train_y, val_y = train_test_split(
        X_train_processed, y_train, test_size=0.2, stratify=y_train, random_state=1
    )

    model = RandomForestClassifier(class_weight="balanced", random_state=1)

    param = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 15),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical(
            "max_features", ["sqrt", "log2", None]
        ),
    }

    model.fit(
        train_x,
        train_y,
    )

    preds = model.predict(val_x)
    f1 = f1_score(val_y, preds.round())

    return f1


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)

print(
    f"Best trial: {study.best_trial.value:.3f} with params: {study.best_trial.params}"
)

sk_rf_model = RandomForestClassifier(
    **study.best_trial.params,
    class_weight="balanced",
    random_state=1,
)

vectorizer = TableVectorizer()

X_train_processed = vectorizer.fit_transform(X_train, y_train)
X_test_processed = vectorizer.transform(X_test)

sk_rf_model.fit(X_train_processed, y_train)
sk_rf_params = sk_rf_model.get_params()

train_score = f1_score(y_train, sk_rf_model.predict(X_train_processed))
test_score = f1_score(y_test, sk_rf_model.predict(X_test_processed))
print(f"Train score: {train_score:.2f}")
print(f"Test score: {test_score:.2f}")

[I 2026-02-27 13:51:27,245] A new study created in memory with name: no-name-c87bf1f0-9e9a-4ded-8d51-99f3a1c5467d
[I 2026-02-27 13:51:30,806] Trial 0 finished with value: 0.6915254237288135 and parameters: {'n_estimators': 402, 'max_depth': 15, 'min_samples_split': 4, 'min_samples_leaf': 4, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6915254237288135.
[I 2026-02-27 13:51:34,792] Trial 1 finished with value: 0.6825938566552902 and parameters: {'n_estimators': 295, 'max_depth': 5, 'min_samples_split': 8, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6915254237288135.
[I 2026-02-27 13:51:38,086] Trial 2 finished with value: 0.6802721088435374 and parameters: {'n_estimators': 400, 'max_depth': 8, 'min_samples_split': 10, 'min_samples_leaf': 12, 'max_features': 'log2'}. Best is trial 0 with value: 0.6915254237288135.
[I 2026-02-27 13:51:41,802] Trial 3 finished with value: 0.6870748299319728 and parameters: {'n_estimators': 403, 'max_depth': 15, 'm

Best trial: 0.702 with params: {'n_estimators': 210, 'max_depth': 9, 'min_samples_split': 12, 'min_samples_leaf': 4, 'max_features': 'sqrt'}
Train score: 0.80
Test score: 0.72


In [30]:
print_resultados_skrub(sk_rf_model, X_test_processed, y_test)

reporte de clasificación:
              precision    recall  f1-score   support

           0       0.95      0.92      0.94      2126
           1       0.67      0.79      0.72       446

    accuracy                           0.90      2572
   macro avg       0.81      0.85      0.83      2572
weighted avg       0.90      0.90      0.90      2572
 

matriz de confusión:
[[1948  178]
 [  92  354]] 

10 características más importantes:
actualizacion       0.186592
ctasahorros         0.137324
s_intereses         0.129949
curtotalingresos    0.088720
puntaje_data        0.069153
valorgarantia       0.064892
tipoasociado        0.061050
vinculacion         0.060561
aportes             0.048338
v_cuota             0.032859
dtype: float64


# Guardado de modelos y parámetos

In [31]:
# import json

# import joblib

# MODELS_DIR = Path().resolve().parent / "models"
# models = {
#     "lgb": (lgb_model, lgb_params),
#     "xgb": (xgb_model, xgb_params),
#     "rf": (rf_model, rf_params),
#     "lr": (lr_model, lr_params),
#     "sk_lgb": (sk_lgb_model, sk_lgb_params),
#     "sk_xgb": (sk_xgb_model, sk_xgb_params),
#     "sk_rf": (sk_rf_model, sk_rf_params),
#     "sk_lr": (sk_lr_model, sk_lr_params),
#     }

# # Guardar modelos en formato joblib y parámetros en formato JSON    

# for name, (model, params) in models.items():
#     joblib.dump(model, MODELS_DIR / f"{name}_model.joblib")
#     with open(MODELS_DIR / f"{name}_params.json", "w") as f:
#         json.dump(params, f)